## Demo synth

Receives incoming MIDI data from the plugin and outputs a simple sine wave based on the note/velocity.

In [ ]:
from pyphonic.src.pyphonic import Pyphonic, midi_parser

In [ ]:
import math

import numpy as np

In [ ]:
class MySynth:
    def __init__(self):
        self.bpm = 128
        self.chunk = 512
        self.freq = None
        self.playing = False
    def __call__(self, audio_data, midi_data, bpm=128, chunk_size=512):
        self.bpm = bpm
        self.chunk_size = chunk_size
        audio_data = np.zeros_like(audio_data)
        midi_messages = midi_parser(midi_data)
        for msg in midi_messages:
            if msg.type == 'note_on':
                self.freq = (440 / 32) * (2 ** ((msg.note - 9) / 12))
                self.angle_delta = (self.freq / 44100) * (2 * math.pi)
                self.playing = True
                self.level = msg.velocity / 255.
                self.current_angle = 0.
            if msg.type == 'note_off':
                self.freq = None
                self.playing = False
                self.angle_delta = 0.
        for i in range(self.chunk):
            if self.playing:
                cur_samp = math.sin(self.current_angle) * self.level
                audio_data[:, i] = cur_samp
                self.current_angle += self.angle_delta
        return audio_data, midi_data

In [ ]:
t = Pyphonic(MySynth)

In [ ]:
t.connect() # Will raise if plugin isn't running

In [ ]:
t.go() # Interrupt this notebook or quit the plugin and it'll stop

In [ ]:
t.disconnect()